# Guide: Data Management Strategy

In [1]:
import polars as pl
import gc
import os

import duckdb

con = duckdb.connect()
con.execute("SET memory_limit='3GB';")
con.execute("SET threads=2;")
con.execute("SET temp_directory='/work/data/tmp';")

## Data Overview & Processing Strategy

### Available Data

We are working with transactional retail data delivered as raw CSV files and converted to Parquet format for efficient analysis.

### Dimension Tables (Small, Fully In-Memory Safe)
- `dim_store_v2`
- `dim_product_v2`
- `dim_campaign_v2`
- `dim_member_v2`

These contain descriptive attributes (e.g., store region, product category, campaign details) and are small enough to load entirely into memory.

### Fact Tables (Large)
- `fact_sales_v2` (~12.5M rows)
- `fact_campaign_sales_v2`
- `fact_points_v2`

These contain transactional data (sales, quantities, revenue, discounts, campaign activity).

### Web Sessions Data (Very Large)
- Web session–level behavioral data (larger than the transactional tables)
- Too large to fully load into memory safely

All datasets have been converted to **Parquet format** to enable:
- Faster reads
- Lower memory usage
- Column pruning
- Efficient analytical queries

Raw CSV files are treated as archival and are no longer used for analysis.

#### We will only scan the dataframe, and not load them into memory. 

### `pl.read_parquet()`

- Loads the full dataset into memory immediately  
- Uses RAM instantly  
- Returns a `DataFrame`  
- Good for small or medium-sized tables  
- Risky for large tables in constrained environments (e.g., 5GB RAM)  


### `pl.scan_parquet()`

- Registers the dataset lazily (does not load immediately)  
- Uses almost no memory at registration time  
- Returns a `LazyFrame`  
- Builds an optimized query plan  
- Only loads data when `.collect()` is called  
- Recommended for large tables and complex transformations  


In [2]:
parquet_dir = "/work/data/parquets"

# Lazy register everything except web sessions
fact_sales = pl.scan_parquet(os.path.join(parquet_dir, "fact_sales_v2.parquet"))
fact_campaign_sales = pl.scan_parquet(os.path.join(parquet_dir, "fact_campaign_sales_v2.parquet"))
fact_points = pl.scan_parquet(os.path.join(parquet_dir, "fact_points_v2.parquet"))

dim_store = pl.scan_parquet(os.path.join(parquet_dir, "dim_store_v2.parquet"))
dim_product = pl.scan_parquet(os.path.join(parquet_dir, "dim_product_v2.parquet"))
dim_campaign = pl.scan_parquet(os.path.join(parquet_dir, "dim_campaign_v2.parquet"))
dim_member = pl.scan_parquet(os.path.join(parquet_dir, "dim_member_v2.parquet"))

This does NOT load data into memory immediately \- and it costs basically zero RAM\. The only difference is that we then have to Perform Transformations Lazily and Materialize Only Final Results

Like this:

In [6]:
result = (
    fact_sales
    .filter(pl.col("sales_date") >= pl.date(2024, 1, 1))
    .group_by("store_id")
    .agg(pl.col("revenue").sum())
)

final_df = result.collect()
final_df

<hr>

## Processing Strategy

### General Rule

<li style="background-color:#ffec99; padding:4px 8px; border-radius:6px;">
- Use Polars for dataframe-style analysis (fast, memory efficient).
</li>

<li style="background-color:#ffec99; padding:4px 8px; border-radius:6px;">
- Use DuckDB for very large datasets (e.g., web sessions) to create subsets before loading into Polars.
</li>
<li style="padding:4px 8px; border-radius:6px;"> - Only load fully into memory when safe.</li>
<li style="padding:4px 8px; border-radius:6px;"> - Perform heavy aggregations before modeling.</li>

## Working with Polars (Primary Workflow)

Polars can handle the main fact tables (~12.5M rows) fully in memory.

### Example: Aggregation

In [ ]:
sales_by_store = (
    fact_sales
    .group_by("store_id")
    .agg(pl.col("revenue").sum())
    .collect()
)

sales_by_store

### Example: Join with Dimension Table

In [ ]:
'''
sales_with_region = sales.join(
    dim_store,
    on="store_id",
    how="left"
)
'''

## Working with DuckDB (Web Sessions)

- The web sessions is too large to load fully into memory.
- Therefore, we first create a filtered or aggregated subset using DuckDB.

### Example: Create Aggregated Subset and Load into Polars

In [ ]:
subset = con.execute("""
    SELECT
        matas_id,
        COUNT(*) AS session_count,
        COUNT(DISTINCT session_id) AS unique_sessions
    FROM '/work/data/parquets/fact_web_sessions_v2.parquet'
    WHERE event_date >= '2024-01-01'
    GROUP BY matas_id
""").arrow()

web_sessions_subset = pl.from_arrow(subset)
web_sessions_subset

And then once we are done with this subset, we can clear it from memory:

In [ ]:
del web_sessions_subset
gc.collect()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=1615b33a-d424-41db-9b01-6976e7db6ad0' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>